<a href="https://colab.research.google.com/github/cchen744/olist-regional-customer-experience-analysis/blob/main/03_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Modeling

1. Objective

* Define the modeling goal clearly:

  * Predict the probability of `is_negative_review`
  * Identify key drivers of customer dissatisfaction
* Framing:

  * This is both a **prediction task** and an **driver analysis task** for business decisions



2. Dataset Construction

* Unit of analysis: (e.g., order-level)
* Target variable:

  * `is_negative_review` (binary)
* Feature groups:

  * Delivery performance
  * Spatial features
  * Product / order features



3. Train / Test Split

* Method:

  * (random split OR time-based split — specify)
* Rationale:

  * Ensure realistic evaluation and avoid data leakage



4. Baseline Models

* Dummy baseline:

  * Predict majority class / average probability
* Logistic Regression:

  * Used as interpretable baseline



5. Feature Processing

* Numerical variables:

  * (e.g., standardization if applied)
* Categorical variables:

  * One-hot encoding



6. Model Training

* Interpretable model:

  * Logistic Regression
* Predictive model:

  * Random Forest



7. Model Evaluation

* Metrics:

  * ROC-AUC (primary)
  * Precision / Recall
  * Accuracy (secondary)
* Considerations:

  * Class imbalance
  * Trade-off between precision and recall (justify which matters more)



8. Model Interpretation

* Logistic Regression:

  * Coefficient analysis → direction and magnitude of effects
* Random Forest:

  * Feature importance → relative contribution of variables
* Key question:

  * What factors drive negative reviews?



9. Spatial Interpretation (Location Intelligence)

* Map predicted probabilities or aggregated risk
* Identify:

  * High-risk regions
  * Spatial patterns of dissatisfaction
* Goal:

  * Translate model outputs into **location-based decisions**


10. Business Implications

* Connect model insights to actions:

  * Logistics optimization
  * Pricing / freight adjustments
  * Regional operational strategies


## Import libraries

In [1]:
# core data handling
import pandas as pd
import numpy as np

# data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# modeling
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)

# geo
import geopandas as gpd

## Dataset construction

In [8]:
# !git clone https://github.com/cchen744/olist-regional-customer-experience-analysis.git
final_df = pd.read_csv('olist-regional-customer-experience-analysis/datasets/final_df.csv')
display(final_df.head())

,order_id,delivery_days,delay_days,late_delivery,same_state,shipping_distance,seller_density,total_price,freight_ratio,num_items,negative_review
0,e481f51cbdc54678b7cc49136f2d6af7,8,-8,False,True,18.576110,1849,29.99,0.290764,1,False
1,53cdb2fc8bc7dce0b6741e2150273451,13,-6,False,False,851.495069,1849,118.70,0.191744,1,False
2,47770eb9100c2d0c44946d9cf07ec65d,9,-18,False,False,514.410666,1849,159.90,0.120200,1,False
3,949d5b44dbf5de918fe9c16f97b45f8a,13,-13,False,False,1822.226336,244,45.00,0.604444,1,False
4,ad21c59c0840e6cb83a9ceb5573f8159,2,-10,False,True,29.676625,1849,19.90,0.438191,1,False


In [9]:
df = final_df.copy()

df['log_price'] = np.log1p(df['total_price'])

y = df['negative_review']
X = df[['delay_days',
        'late_delivery',
        'same_state',
        'seller_density',
        'log_price',
        'freight_ratio']]

X.isna().sum()
y.isna().sum()

np.int64(0)

## Train/Test Split

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Baseline Models

### Dummy baseline

In [26]:
dummy = DummyClassifier(strategy='most_frequent')

dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)

print("Dummy Accuracy:", accuracy_score(y_test, y_pred_dummy))
print("Dummy Recall:", recall_score(y_test, y_pred_dummy))

Dummy Accuracy: 0.8717667918231122
Dummy Recall: 0.0


In [27]:
y.value_counts(normalize=True)

,proportion
negative_review,
False,0.871782
True,0.128218


The dummy model already achieves high accuracy because the dataset is imbalanced: negative reviews are relatively rare. In this setting, accuracy is not a sufficient evaluation metric. We care more about recall, since failing to identify a negative review (false negative) is more costly than incorrectly flagging a non-negative review as negative.

In [28]:
y_prob_dummy = dummy.predict_proba(X_test)[:, 1]
print("Dummy AUC:", roc_auc_score(y_test, y_prob_dummy))

Dummy AUC: 0.5


### Logistic Regression

In [29]:
lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Logistic Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Logistic Recall:", recall_score(y_test, y_pred_lr))

Logistic Accuracy: 0.8864205256570713
Logistic Recall: 0.31354209028060187


In [36]:
y_prob_lr = lr.predict_proba(X_test)[:, 1]
print("Logistic AUC:", roc_auc_score(y_test, y_prob_lr))

Logistic AUC: 0.693626038302918


In [30]:
# Random Forest
rf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(random_state=42))
])

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Random Forest Recall:", recall_score(y_test, y_pred_rf))

Random Forest Accuracy: 0.882717980809345
Random Forest Recall: 0.27816185441236274


In [31]:
y_prob_rf = rf.predict_proba(X_test)[:, 1]
print("RF AUC:", roc_auc_score(y_test, y_prob_rf))

RF AUC: 0.6937683373114374


Compared with the dummy baseline, Logistic Regression and Random Forest show only a modest improvement in accuracy, but a substantial improvement in recall and ROC-AUC. This suggests that the model is capturing meaningful patterns related to negative reviews, rather than simply predicting the majority class.

## Modeling & Evaluation

In [32]:
!pip install xgboost

In [35]:
# XGBoost
from xgboost import XGBClassifier

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
print(neg / pos)

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=neg/pos,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

print("XGB Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("XGB Recall:", recall_score(y_test, y_pred_xgb))
print("XGB AUC:", roc_auc_score(y_test, y_prob_xgb))

6.799471222290014
XGB Accuracy: 0.8177930746766792
XGB Recall: 0.4656364375762505
XGB AUC: 0.7122216931706425


In [37]:
xgb.feature_importances_

array([0.7549217 , 0.        , 0.04094839, 0.05152991, 0.06655298,
       0.08604707], dtype=float32)

## Interpretation

| Model                | Accuracy | Recall | ROC-AUC | Business Interpretation                          |
|---------------------|----------|--------|---------|--------------------------------------------------|
| Dummy               | 0.872    | 0.000  | 0.500   | Fails to detect any negative reviews             |
| Logistic Regression | 0.886    | 0.314  | 0.694   | Captures basic logistic drivers (delay, freight) |
| Random Forest       | 0.883    | 0.278  | 0.694   | Limited gain from non-linear bagging             |
| XGBoost             | 0.818    | 0.466  | 0.712   | Best detection of at-risk orders                 |

### Model Performance

Compared with baseline models, XGBoost achieves the best overall performance. While improvements in ROC-AUC are modest (from ~0.69 to ~0.71), the model shows a substantial increase in recall, rising from approximately 0.31 (Logistic Regression) to 0.47.

This indicates that XGBoost is significantly more effective at identifying negative reviews, which is especially important in this imbalanced setting where missing negative cases (false negatives) is more costly than false positives.



### Key Drivers of Negative Reviews

Feature importance analysis from XGBoost reveals that `delay_days` is by far the most influential variable, accounting for the majority of model importance. This highlights delivery delay as the primary driver of customer dissatisfaction.

Other variables such as `freight_ratio`, `log_price`, and `seller_density` contribute to a lesser extent, suggesting that cost perception and regional logistics conditions also play a role. In contrast, binary indicators such as `late_delivery` provide limited additional signal beyond what is already captured by continuous variables.

Overall, the results indicate that customer dissatisfaction is primarily driven by **delivery performance and cost-related factors**, rather than simple categorical attributes.



### Limitations

Despite the improvement achieved by XGBoost, the overall ROC-AUC remains around 0.71, indicating that predictive performance is constrained by the available features.

Many important drivers of customer dissatisfaction—such as product quality, customer expectations, and service experience—are not captured in the current dataset. As a result, logistics-related features alone are insufficient to fully explain negative reviews.



### Business Implications

The findings suggest that improving delivery reliability should be the top priority for reducing customer dissatisfaction, as delivery delay is the dominant driver of negative reviews.

In addition, the influence of freight-related variables indicates that customer perception of cost also affects satisfaction. Aligning delivery performance with customer expectations—both in terms of timing and pricing—is therefore critical.

From an operational perspective, the model can be used to identify at-risk orders in advance, enabling targeted interventions such as expedited shipping or proactive customer communication.



### Conclusion

Overall, the results demonstrate that while advanced models such as XGBoost can improve the detection of negative reviews, the primary limitation lies in feature availability rather than model choice.

This highlights the importance of integrating richer behavioral or product-level data to further improve predictive performance in future work.


Feature importance from XGBoost shows that `delay_days` is by far the most influential factor in predicting negative reviews, accounting for the majority of model importance. Other variables such as `freight_ratio`, `log_price`, and `seller_density` contribute to a lesser extent, while binary indicators like `late_delivery` add limited additional signal.

This suggests that customer dissatisfaction is primarily driven by delivery performance rather than static attributes or simple categorical indicators.